# M14 — Discover Structure Without Labels

**Objective:** use clustering to investigate structure without a target label. The engineering loop is `unlabelled rows → feature contract → geometry → candidate partitions → internal diagnostics → cautious interpretation`.

This is not a hunt for hidden true classes. Every result depends on selected features, scaling, Euclidean distance, K-means assumptions, the observed sample, and `k`. Record each prediction in your own notes before running the experiment that follows it.

## 1. Load and audit the unlabelled fixture

**Prediction before action:** Which column must never enter a distance calculation? Which selected feature do you expect to have the widest raw numeric range?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_samples, silhouette_score
from sklearn.preprocessing import StandardScaler

SEED = 14
np.random.seed(SEED)
pd.set_option("display.max_columns", 20)

DATA_CANDIDATES = [
    Path("datasets/M14/learning_sessions.csv"),
    Path("../datasets/M14/learning_sessions.csv"),
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find the local datasets/M14/learning_sessions.csv fixture.")

sessions = pd.read_csv(DATA_PATH)
SELECTED_FEATURES = [
    "active_minutes",
    "practice_ratio",
    "review_ratio",
    "help_requests",
    "context_switches",
    "completion_fraction",
    "activity_events",
]
FORBIDDEN_LABEL_TOKENS = ("target", "label", "class", "segment", "cluster", "cohort")

assert sessions.shape == (54, 8)
assert sessions["session_id"].is_unique
assert all(feature in sessions.columns for feature in SELECTED_FEATURES)
assert not any(token in column.lower() for column in sessions.columns for token in FORBIDDEN_LABEL_TOKENS)
assert not sessions[SELECTED_FEATURES].isna().any().any()
assert np.isfinite(sessions[SELECTED_FEATURES].to_numpy(dtype=float)).all()

feature_frame = sessions[SELECTED_FEATURES].copy()
feature_audit = pd.DataFrame({
    "dtype": feature_frame.dtypes.astype(str),
    "minimum": feature_frame.min(),
    "maximum": feature_frame.max(),
    "range": feature_frame.max() - feature_frame.min(),
    "std": feature_frame.std(ddof=0),
})
print(f"rows={len(sessions)}, target columns=0, identifier excluded=session_id")
feature_audit

## 2. Feature and distance contract

`session_id` is traceability metadata, not a feature. The seven numeric columns are retained for this investigation because each could describe session behavior or instrumentation. K-means minimizes squared Euclidean distance to centers, so units and variance directly define the geometry. Equal inclusion does **not** mean equal influence on raw values.

**Prediction before action:** Will the raw `k=3` partition follow behavioral profiles or the high-magnitude event counter? What evidence would distinguish the two?

In [ ]:
def squared_distance_contributions(matrix, names):
    pairwise_differences = matrix[:, None, :] - matrix[None, :, :]
    contributions = np.square(pairwise_differences).sum(axis=(0, 1))
    return pd.Series(contributions / contributions.sum(), index=names, name="raw_distance_share")

X_raw = feature_frame.to_numpy(dtype=float)
raw_model = KMeans(n_clusters=3, init="k-means++", n_init=20, random_state=SEED)
raw_labels = raw_model.fit_predict(X_raw)
raw_silhouette = silhouette_score(X_raw, raw_labels)
raw_distance_share = squared_distance_contributions(X_raw, SELECTED_FEATURES).sort_values(ascending=False)

raw_profile = (
    sessions.assign(raw_cluster=raw_labels)
    .groupby("raw_cluster")
    .agg(count=("session_id", "size"), mean_events=("activity_events", "mean"), mean_active_minutes=("active_minutes", "mean"))
    .round(2)
)
assert raw_distance_share.index[0] == "activity_events"
print(f"raw k=3 silhouette={raw_silhouette:.3f}")
display(raw_distance_share.to_frame(), raw_profile)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(sessions["activity_events"], sessions["active_minutes"], c=raw_labels, cmap="viridis", s=45)
ax.set(xlabel="activity_events (dominant raw scale)", ylabel="active_minutes", title="A convincing raw partition can be a scale artifact")
plt.show()

## 3. Standardize the selected features

StandardScaler removes each selected feature's mean and scales it to unit variance. This is a consequential choice, not housekeeping: it changes distance and gives each standardized feature comparable variance. It is also sensitive to outliers.

**Prediction before action:** How much will the raw and standardized `k=3` assignments agree? A high raw silhouette does not answer this question.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(feature_frame)
scaled_check = pd.DataFrame(X_scaled, columns=SELECTED_FEATURES).agg(["mean", "std"]).round(3)
scaled_k3_labels = KMeans(n_clusters=3, init="k-means++", n_init=20, random_state=SEED).fit_predict(X_scaled)
raw_scaled_agreement = adjusted_rand_score(raw_labels, scaled_k3_labels)

assert np.allclose(X_scaled.mean(axis=0), 0.0, atol=1e-12)
assert np.allclose(X_scaled.std(axis=0, ddof=0), 1.0, atol=1e-12)
print(f"raw-vs-scaled k=3 adjusted Rand agreement={raw_scaled_agreement:.3f}")
scaled_check

## 4. Choose `k` with multiple internal diagnostics

**Prediction before action:** For `k=2..6`, which candidate will have the strongest mean silhouette? Which candidate might fragment into small clusters? Which diagnostics should veto an otherwise attractive score?

Inertia must fall as `k` increases, so the smallest inertia is not a selection rule. Silhouette measures cohesion/separation in the chosen geometry. Cross-seed adjusted Rand agreement detects initialization sensitivity but cannot establish external truth.

In [ ]:
def evaluate_candidate(k, matrix, seed=SEED):
    primary = KMeans(n_clusters=k, init="k-means++", n_init=20, random_state=seed)
    labels = primary.fit_predict(matrix)
    repeat_labels = KMeans(
        n_clusters=k, init="k-means++", n_init=20, random_state=seed + 1000 + k
    ).fit_predict(matrix)
    sizes = np.bincount(labels, minlength=k)
    return {
        "k": k,
        "inertia": primary.inertia_,
        "silhouette": silhouette_score(matrix, labels),
        "min_cluster_size": int(sizes.min()),
        "stability_ari": adjusted_rand_score(labels, repeat_labels),
    }

diagnostics = pd.DataFrame([evaluate_candidate(k, X_scaled) for k in range(2, 7)])
MIN_ACCEPTABLE_CLUSTER_SIZE = 5
MIN_ACCEPTABLE_STABILITY = 0.90
eligible = diagnostics.loc[
    (diagnostics["min_cluster_size"] >= MIN_ACCEPTABLE_CLUSTER_SIZE)
    & (diagnostics["stability_ari"] >= MIN_ACCEPTABLE_STABILITY)
]
selected_k = int(eligible.loc[eligible["silhouette"].idxmax(), "k"])

assert diagnostics["inertia"].is_monotonic_decreasing
assert selected_k == 2, "The deterministic fixture's diagnostic baseline changed; inspect before interpreting."
print(
    f"guardrails: min size >= {MIN_ACCEPTABLE_CLUSTER_SIZE}, "
    f"stability >= {MIN_ACCEPTABLE_STABILITY:.2f}"
)
print(f"highest-silhouette eligible candidate: k={selected_k}")
diagnostics.round(3)

The fixture's largest mean silhouette occurs at `k=2`; that is an internal-diagnostic candidate, not a discovered natural number of groups. Record the competing evidence in the table. In particular, another candidate may be operationally interesting or stable while having weaker separation, and the search range itself was chosen by us.

## 5. Centers and within-cluster structure

**Prediction before action:** Which original-unit center features will differ most? Which cluster will contain the weakest-assigned observations? Mean silhouette can conceal negative samples and center distance can expose unusual points.

In [ ]:
final_model = KMeans(n_clusters=selected_k, init="k-means++", n_init=20, random_state=SEED)
final_labels = final_model.fit_predict(X_scaled)
center_profiles = pd.DataFrame(
    scaler.inverse_transform(final_model.cluster_centers_), columns=SELECTED_FEATURES
).round(3)
center_profiles.index.name = "cluster_id"

sample_silhouettes = silhouette_samples(X_scaled, final_labels)
assigned_distances = np.linalg.norm(X_scaled - final_model.cluster_centers_[final_labels], axis=1)
cluster_results = sessions[["session_id"]].copy()
cluster_results["cluster_id"] = final_labels
cluster_results["silhouette"] = sample_silhouettes
cluster_results["distance_to_center"] = assigned_distances

within_cluster = cluster_results.groupby("cluster_id").agg(
    count=("session_id", "size"),
    mean_silhouette=("silhouette", "mean"),
    min_silhouette=("silhouette", "min"),
    mean_center_distance=("distance_to_center", "mean"),
    max_center_distance=("distance_to_center", "max"),
)
farthest_observations = cluster_results.nlargest(6, "distance_to_center").merge(sessions, on="session_id")

assert len(cluster_results) == len(sessions)
assert within_cluster["count"].sum() == len(sessions)
display(center_profiles, within_cluster.round(3), farthest_observations.round(3))

In [ ]:
fig, axes = plt.subplots(1, selected_k, figsize=(10, 3), sharey=True)
axes = np.atleast_1d(axes)
for cluster_id, ax in enumerate(axes):
    values = np.sort(sample_silhouettes[final_labels == cluster_id])
    ax.barh(np.arange(len(values)), values, color=plt.cm.viridis(cluster_id / max(selected_k - 1, 1)))
    ax.axvline(sample_silhouettes.mean(), color="black", linestyle="--", linewidth=1)
    ax.axvline(0, color="red", linewidth=1)
    ax.set(title=f"cluster {cluster_id}", xlabel="sample silhouette")
axes[0].set_ylabel("observations, sorted")
fig.suptitle("Within-cluster evidence hidden by the mean")
fig.tight_layout()
plt.show()

## 6. Controlled failure: arbitrary `k` and one corrupted outlier

**Prediction before action:** What will forcing `k=5` split? If one corrupted record is extreme across the measurement system, could a better silhouette actually describe a worse answer? Predict cluster sizes and assignment agreement before running.

In [ ]:
forced_k = 5
forced_labels = KMeans(n_clusters=forced_k, init="k-means++", n_init=20, random_state=SEED).fit_predict(X_scaled)
forced_result = {
    "k": forced_k,
    "silhouette": silhouette_score(X_scaled, forced_labels),
    "cluster_sizes": np.bincount(forced_labels, minlength=forced_k).tolist(),
}

corrupted = sessions.copy()
corrupted.loc[len(corrupted)] = {
    "session_id": "LS_CORRUPTED",
    "active_minutes": 250,
    "practice_ratio": 0.05,
    "review_ratio": 0.95,
    "help_requests": 30,
    "context_switches": 50,
    "completion_fraction": 0.10,
    "activity_events": 60000,
}
corrupted_scaler = StandardScaler()
X_corrupted = corrupted_scaler.fit_transform(corrupted[SELECTED_FEATURES])
corrupted_labels = KMeans(
    n_clusters=selected_k, init="k-means++", n_init=20, random_state=SEED
).fit_predict(X_corrupted)
outlier_agreement = adjusted_rand_score(final_labels, corrupted_labels[:-1])
outlier_silhouette = silhouette_score(X_corrupted, corrupted_labels)
outlier_sizes = np.bincount(corrupted_labels, minlength=selected_k)

assert outlier_sizes.min() == 1
assert outlier_agreement < 0.1
display(
    pd.DataFrame([forced_result]),
    pd.DataFrame([{
        "scenario": "corrupted outlier included",
        "k": selected_k,
        "silhouette": outlier_silhouette,
        "agreement_on_unchanged_rows": outlier_agreement,
        "cluster_sizes": outlier_sizes.tolist(),
        "outlier_cluster": int(corrupted_labels[-1]),
    }]).round(3),
)

The stress test is deliberately severe: one corrupted record can consume an entire center, collapse structure among the unchanged rows, and still increase mean silhouette. Diagnose the data-quality mechanism before choosing deletion, clipping, robust scaling, a different algorithm, or retaining the point. Improving a score is not by itself a defensible outlier policy.

## 7. Visualization is a lossy view

**Prediction before action:** How much standardized variance will two principal components retain? What claim would remain unsupported even if the colors separate perfectly?

In [ ]:
pca = PCA(n_components=2)
projected = pca.fit_transform(X_scaled)
retained_variance = float(pca.explained_variance_ratio_.sum())

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(projected[:, 0], projected[:, 1], c=final_labels, cmap="viridis", s=50)
ax.set(
    xlabel="principal component 1",
    ylabel="principal component 2",
    title=f"Display only: two PCs retain {retained_variance:.1%} of variance",
)
plt.show()
print(f"full-space silhouette={silhouette_score(X_scaled, final_labels):.3f}")
print(f"two-component retained variance={retained_variance:.3f}")

## 8. Interpretation boundary: clusters are not true classes

The integer `cluster_id` values are arbitrary names. They are not ordered levels, diagnoses, learner identities, or externally verified classes. The center tables summarize this fixture under one declared geometry. They can generate questions; they cannot establish causes or justify interventions.

Before any downstream use, define an external validation question, check stability across samples and time, review proxy and harm risks, and record the consequential choices in `missions/M14/adr_prompt.md`.

## 9. Evidence, ADR, and no-AI transfer

Submit the artifacts in `missions/M14/evidence_contract.yaml`. Complete the required ADR from `missions/M14/adr_prompt.md`, including evidence against your selection and reversal triggers. Then close this notebook and complete `missions/M14/no_ai_gate.md` on a fresh unlabelled dataset without AI-generated code or prose.

Mission completion means you can defend and challenge an unsupervised result—not that you found the right labels.